<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/Module1_Labs(v2)/Lab3_Pauli_Rotation_Operators.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# QOS Lab 3 — Pauli & Rotation Operators
### Quantum Optimization and Simulation | Cleveland State University
**Instructor:** Prof. Chansu Yu | Washkewicz College of Engineering

---
## Learning Objectives
1. Understand how the **Pauli-Z** gate encodes a binary phase flip
2. Understand the **rotation operator $R_Z(\theta)$** and how it encodes a *continuous* phase
3. Connect Euler's formula $e^{i\theta} = \cos\theta + i\sin\theta$ to quantum gates
4. Verify that $R_Z$ does **not** change measurement probabilities — only phases
5. See why $R_Z$ is the building block of QAOA's **Cost Operator** (Step 2: Judges' Scores)

---
### 📖 Background: Phase and Rotation

Recall from Lab 2:
- The Pauli-Z gate flips the sign (phase) of the $|1\rangle$ component:
$$Z|\psi\rangle = Z(a|0\rangle + b|1\rangle) = a|0\rangle - b|1\rangle$$
- This is a **180° rotation** around the z-axis on the Bloch sphere

The **rotation operator** generalizes this to any angle $\theta$:
$$R_Z(\theta) = e^{-i\frac{\theta}{2}Z} = \begin{bmatrix} e^{-i\theta/2} & 0 \\ 0 & e^{+i\theta/2} \end{bmatrix}$$

Key property: $R_Z$ **does not change probabilities** — only phases!
$$R_Z(\theta)|\psi\rangle: \text{ the } |0\rangle \text{ component rotates by } -\theta/2, \text{ the } |1\rangle \text{ component by } +\theta/2$$

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

simulator = AerSimulator()

# Standard gates
I = np.eye(2, dtype=complex)
X = np.array([[0,1],[1,0]], dtype=complex)
Y = np.array([[0,-1j],[1j,0]], dtype=complex)
Z = np.array([[1,0],[0,-1]], dtype=complex)
H = (1/np.sqrt(2)) * np.array([[1,1],[1,-1]], dtype=complex)

ket_0 = np.array([1.0+0j, 0.0+0j])
ket_1 = np.array([0.0+0j, 1.0+0j])
plus  = np.array([1/np.sqrt(2)+0j, 1/np.sqrt(2)+0j])

print("Setup complete.")

---
## Part 1: Euler's Formula and Complex Phases

In [ ]:
# ── 1.1  Euler's formula: e^{iθ} = cos θ + i sin θ ───────────────────────────
# Think of e^{iθ} as a point on the unit circle in the complex plane
# at angle θ from the positive real axis

thetas = [0, np.pi/4, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
labels = ['0', 'π/4', 'π/2', 'π', '3π/2', '2π']

print(f"{'θ':>6} | {'e^(iθ) = cos+i·sin':>30} | {'|e^(iθ)|':>10}")
print("-" * 55)
for theta, label in zip(thetas, labels):
    val  = np.exp(1j * theta)
    mag  = abs(val)
    print(f"{label:>6} | {val.real:+.4f} + {val.imag:+.4f}i      | {mag:.4f}")

print("\nKey: |e^(iθ)| = 1 always — it's ALWAYS on the unit circle!")

In [ ]:
# ── 1.2  Visualize the unit circle ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5,5))

# Unit circle
theta_range = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(theta_range), np.sin(theta_range), 'lightgray', lw=1)
ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)

# Special angles
special = [(0,'0°','red'), (np.pi/4,'45°','blue'), (np.pi/2,'90°','green'),
           (np.pi,'180°','orange'), (3*np.pi/2,'270°','purple')]

for theta, label, color in special:
    x, y = np.cos(theta), np.sin(theta)
    ax.annotate('', xy=(x,y), xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    ax.text(x*1.15, y*1.15, f'e^(i·{label})', ha='center', fontsize=8, color=color)

ax.set_xlim(-1.5,1.5); ax.set_ylim(-1.5,1.5)
ax.set_aspect('equal')
ax.set_xlabel('Real part (cos θ)')
ax.set_ylabel('Imaginary part (sin θ)')
ax.set_title('Unit circle: e^(iθ) = cos θ + i·sin θ')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 2: The R_Z(θ) Rotation Operator

In [ ]:
# ── 2.1  Build R_Z(θ) as a matrix ────────────────────────────────────────────
def Rz_matrix(theta):
    """R_Z(θ) = [[e^{-iθ/2}, 0], [0, e^{+iθ/2}]]"""
    return np.array([[np.exp(-1j*theta/2), 0],
                     [0, np.exp(+1j*theta/2)]])

# Test: R_Z(180°) should equal -i·Z (up to global phase)
Rz_180 = Rz_matrix(np.pi)
print("R_Z(180°) =")
print(np.round(Rz_180, 4))
print("\nExpected: [[-i, 0], [0, i]]")

# Test: R_Z(0°) should equal Identity
Rz_0 = Rz_matrix(0)
print("\nR_Z(0°) =")
print(np.round(Rz_0, 4))
print("Expected: [[1, 0], [0, 1]] = I")

In [ ]:
# ── 2.2  R_Z does NOT change measurement probabilities ────────────────────────
state = plus.copy()  # H|0⟩ = (|0⟩+|1⟩)/√2

print("Starting state: H|0⟩ = |+⟩")
print(f"  amplitudes:    {np.round(state,4)}")
print(f"  P(|0⟩)={abs(state[0])**2:.3f}, P(|1⟩)={abs(state[1])**2:.3f}")

print("\nAfter applying R_Z(θ) for several angles:")
print(f"{'θ':>8} | {'a₀ after Rz':>25} | {'a₁ after Rz':>25} | {'P(|0⟩)':>8} | {'P(|1⟩)':>8}")
print("-" * 90)

for theta_deg in [0, 30, 60, 90, 120, 180]:
    theta = np.radians(theta_deg)
    new_state = Rz_matrix(theta) @ state
    p0 = abs(new_state[0])**2
    p1 = abs(new_state[1])**2
    print(f"{theta_deg:>7}° | {new_state[0]:>25.4f} | {new_state[1]:>25.4f} | {p0:>8.4f} | {p1:>8.4f}")

print("\n⚠️  PROBABILITIES ARE UNCHANGED for all θ!")
print("   But the PHASES change — this is what QAOA exploits for interference.")

In [ ]:
# ── 2.3  Visualize phase evolution of |0⟩ and |1⟩ components ─────────────────
thetas_deg = np.linspace(0, 360, 200)
thetas_rad = np.radians(thetas_deg)

phase_0 = [-t/2 for t in thetas_rad]    # |0⟩ component: rotates by -θ/2
phase_1 = [+t/2 for t in thetas_rad]    # |1⟩ component: rotates by +θ/2
relative_phase = np.degrees([p1-p0 for p0,p1 in zip(phase_0, phase_1)])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(thetas_deg, np.degrees(phase_0), 'b-', label='|0⟩ phase: −θ/2')
axes[0].plot(thetas_deg, np.degrees(phase_1), 'r-', label='|1⟩ phase: +θ/2')
axes[0].set_xlabel('θ (degrees)'); axes[0].set_ylabel('Phase (degrees)')
axes[0].set_title('Individual component phases after R_Z(θ)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(thetas_deg, relative_phase, 'g-', lw=2)
axes[1].set_xlabel('θ (degrees)'); axes[1].set_ylabel('Relative phase (degrees)')
axes[1].set_title('Relative phase between |0⟩ and |1⟩\n= the "out of sync" amount')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("R_Z(θ) creates a relative phase of θ between |0⟩ and |1⟩ components.")

In [ ]:
# ── 2.4  R_Z in Qiskit ──────────────────────────────────────────────────────
theta = np.pi / 2   # 90 degrees

qc_rz = QuantumCircuit(1)
qc_rz.h(0)          # Step 1: create superposition
qc_rz.rz(theta, 0)  # Step 2: apply R_Z(90°)

sv = Statevector(qc_rz)
print(f"R_Z(90°) · H|0⟩ statevector:")
print(f"  a₀ = {sv.data[0]:.4f}   |a₀|² = {abs(sv.data[0])**2:.4f}")
print(f"  a₁ = {sv.data[1]:.4f}   |a₁|² = {abs(sv.data[1])**2:.4f}")
print(f"\nProbabilities: {sv.probabilities_dict()}")
print("Still 50/50 — phase is hidden until interference occurs.")

### ✏️ Exercise 3.1 — R_Z as a Phase Marker

QAOA's Cost Operator uses $R_Z$ to encode a cost value $c$ into phase:
$$R_Z(-2\gamma c)|\psi\rangle$$

Think of $\gamma = \pi/6$ (30 degrees per unit of cost).
For a bitstring with cut value $c$, the phase added is $\gamma \cdot c$.

**Tasks:**
1. For each cut value `c = 0, 1, 2, 3, 4, 5, 6`, compute the relative phase angle (in degrees) that $R_Z(-2\gamma c)$ creates, with $\gamma = \pi/6$.
2. Which cut value produces 0° phase? Which produces 180°?
3. Why does the phase matter if we can't measure it directly?

In [ ]:
# YOUR CODE HERE
gamma = np.pi / 6   # 30 degrees per unit of cost

print("Phase encoding in QAOA Cost Operator:")
print(f"γ = π/6 = {np.degrees(gamma):.1f}° per unit of cut")
print(f"\n{'Cut c':>6} | {'Phase = γ·c (rad)':>18} | {'Phase (degrees)':>16} | {'e^(i·phase)':>25}")
print("-" * 75)

for c in range(7):
    phase_rad = gamma * c
    phase_deg = np.degrees(phase_rad)
    eph = np.exp(1j * phase_rad)
    print(f"{c:>6} | {phase_rad:>18.4f} | {phase_deg:>16.2f} | {eph.real:+.4f} + {eph.imag:+.4f}i")

print("\nAnswers:")
print("  c=0 → 0°  phase (no rotation)")
print("  c=6 → 180° phase (maximum rotation for this γ)")
print("  Phase difference between c=4,6 vs c=0,2 is what enables")
print("  constructive vs destructive interference in the Mixer step.")

---
## Part 3: Connecting R_Z to QAOA's Cost Operator

The QAOA cost operator for one edge $(i,j)$ is:
$$R_{ZZ}(\gamma) = e^{-i\frac{\gamma}{2}Z_iZ_j}$$

For now, let's understand the single-qubit version and see the phase effect directly.

In [ ]:
# ── 3.1  Simulate the cost operator effect on a single qubit ─────────────────
# Simplified: imagine 1 qubit where cut value is encoded
# State after H: (|0⟩ + |1⟩)/√2
# Apply R_Z(γ) to encode different costs for |0⟩ and |1⟩

def show_phase_effect(gamma_val, label):
    qc = QuantumCircuit(1)
    qc.h(0)                     # superposition
    qc.rz(gamma_val, 0)         # encode phase
    sv = Statevector(qc)
    a0, a1 = sv.data[0], sv.data[1]
    phase_diff_deg = np.degrees(np.angle(a1) - np.angle(a0))
    print(f"  {label:20s}: a₀={a0:.3f}, a₁={a1:.3f}, phase_diff={phase_diff_deg:+.1f}°, P(0)={abs(a0)**2:.3f}, P(1)={abs(a1)**2:.3f}")

print("Effect of R_Z(γ) on H|0⟩ for various γ values:")
print(f"  {'Scenario':20s}: a₀, a₁ amplitudes, phase difference, probabilities")
print("-" * 90)
show_phase_effect(0,         "γ=0 (no rotation)")
show_phase_effect(np.pi/6,   "γ=π/6 (30°)")
show_phase_effect(np.pi/3,   "γ=π/3 (60°)")
show_phase_effect(np.pi/2,   "γ=π/2 (90°)")
show_phase_effect(np.pi,     "γ=π (180°)")
show_phase_effect(2*np.pi,   "γ=2π (360°=wrap)")

print("\nKey: Probabilities are always 50/50. Only the phase changes.")
print("When γ is too large (wraps around), the phase ordering gets scrambled.")

In [ ]:
# ── 3.2  Preview: phase → amplitude (what the Mixer does) ────────────────────
# The Mixer (R_X(β)) converts phase differences into probability differences
# Let's see this with a 1-qubit example

gamma = np.pi / 3

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
beta_values = [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]
colors = plt.cm.viridis(np.linspace(0, 1, len(beta_values)))

for ax_idx, (gamma_val, gamma_label) in enumerate([
    (np.pi/6,  'γ=π/6 (small)'),
    (np.pi/3,  'γ=π/3 (medium)'),
    (np.pi/2,  'γ=π/2 (larger)'),
]):
    ax = axes[ax_idx]
    p0_vals = []
    for beta in beta_values:
        qc = QuantumCircuit(1)
        qc.h(0)                  # Step 1: Everyone on stage
        qc.rz(gamma_val, 0)      # Step 2: Judges' scores
        qc.rx(beta, 0)           # Step 3: Crowd's applause (Mixer)
        sv = Statevector(qc)
        p0_vals.append(abs(sv.data[0])**2)

    ax.bar([f'β={np.degrees(b):.0f}°' for b in beta_values], p0_vals,
           color=colors, edgecolor='black')
    ax.axhline(0.5, color='red', linestyle='--', lw=1, label='P=0.5 (no bias)')
    ax.set_ylim(0, 1)
    ax.set_xlabel('Mixer angle β')
    ax.set_ylabel('P(|0⟩)')
    ax.set_title(f'H → R_Z({gamma_label}) → R_X(β)\nPhase→Amplitude conversion')
    ax.legend(fontsize=7)
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()
print("The Mixer (R_X) translates phase differences into probability shifts.")
print("Correct γ + correct β → bias toward the optimal solution.")

---
### ✏️ Exercise 3.2 — R_X (Mixer) Exploration

The Mixer uses $R_X(\beta) = e^{-i\frac{\beta}{2}X}$.

1. Build the matrix for $R_X(\beta)$ using Euler's formula (similar to $R_Z$)
2. Verify your formula matches Qiskit's `qc.rx(beta, 0)` for β = π/2
3. What does $R_X(\pi)$ do? What is its matrix?
4. Apply the full 1-qubit QAOA sequence (H → R_Z(γ) → R_X(β)) for γ=π/3, β=π/4. What is P(|0⟩)?

In [ ]:
# YOUR CODE HERE

# 1. R_X matrix using Euler's formula
# e^{-iβ/2 X} = cos(β/2)·I - i·sin(β/2)·X
def Rx_matrix(beta):
    return np.cos(beta/2)*I - 1j*np.sin(beta/2)*X

print("1. R_X(π/2) matrix (manual):")
print(np.round(Rx_matrix(np.pi/2), 4))

# 2. Verify against Qiskit
qc_rx = QuantumCircuit(1)
qc_rx.rx(np.pi/2, 0)
from qiskit.quantum_info import Operator
qiskit_rx = Operator(qc_rx).data
print("\n2. R_X(π/2) matrix (Qiskit):")
print(np.round(qiskit_rx, 4))
print("   Match:", np.allclose(Rx_matrix(np.pi/2), qiskit_rx))

# 3. R_X(π)
print("\n3. R_X(π) = ?")
print(np.round(Rx_matrix(np.pi), 4))
print("   R_X(π) = -i·X: same as Pauli-X (quantum NOT), up to global phase")

# 4. Full 1-qubit QAOA
gamma, beta = np.pi/3, np.pi/4
qc_qaoa1 = QuantumCircuit(1)
qc_qaoa1.h(0)
qc_qaoa1.rz(gamma, 0)
qc_qaoa1.rx(beta, 0)
sv = Statevector(qc_qaoa1)
print(f"\n4. H → R_Z(γ=60°) → R_X(β=45°):")
print(f"   Statevector: {np.round(sv.data, 4)}")
print(f"   P(|0⟩) = {abs(sv.data[0])**2:.4f}")
print(f"   P(|1⟩) = {abs(sv.data[1])**2:.4f}")
print("   No longer 50/50 — the phase was converted to amplitude bias!")

---
## ✅ Lab 3 Summary

| Concept | Formula | Key Insight |
|---------|---------|-------------|
| Euler's formula | $e^{i\theta} = \cos\theta + i\sin\theta$ | Phase = angle on unit circle |
| Pauli-Z | $Z = \begin{bmatrix}1&0\\0&-1\end{bmatrix}$ | Flips sign of $|1\rangle$: 180° phase |
| $R_Z(\theta)$ | $\begin{bmatrix}e^{-i\theta/2}&0\\0&e^{+i\theta/2}\end{bmatrix}$ | Continuous phase rotation |
| $R_X(\beta)$ | $\cos(\beta/2)I - i\sin(\beta/2)X$ | Converts phase → amplitude (Mixer) |
| Key property | $R_Z$ doesn't change $P(|0\rangle)$ or $P(|1\rangle)$ | Phase is invisible until interference |
| QAOA Step 2 | $R_Z$ encodes cost $c$ as phase $\gamma c$ | "Judges' Scores" |
| QAOA Step 3 | $R_X(\beta)$ converts phase → probability | "Crowd's Applause" |

## 🔭 Preview of Lab 4
Next: **Two-Qubit Gates & Entanglement** — specifically the $R_{ZZ}(\gamma)$ gate that is the actual building block of QAOA's cost operator over graph edges.

---
*QOS Lab 3 | Prof. Chansu Yu | Cleveland State University*